# Lab 4: Profiling and Benchmarking Python Code

 Name:**Mainza Muya**  
 Student number:**202406802**    
 Course:**ICD341 Advanced Statistical Computing**  



## Part A: `%timeit`

In [2]:
import numpy as np

x = np.arange(1_000_000)

In [3]:
%timeit x ** 2

6.1 ms ± 769 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Result recorded:** _write your time here_

## Part B: Loop vs vectorization

In [4]:
def square_loop(x):
    result = []
    for value in x:
        result.append(value ** 2)
    return result

def square_numpy(x):
    return x ** 2

In [5]:
# -o stores the result so we can compute the speed-up; -n 1 -r 3 keeps the slow loop quick
t_loop = %timeit -o -n 1 -r 3 square_loop(x)

525 ms ± 23.6 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [6]:
t_numpy = %timeit -o square_numpy(x)

5.19 ms ± 306 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
print(f"Python loop:         {t_loop.best*1000:10.3f} ms")
print(f"NumPy vectorization: {t_numpy.best*1000:10.3f} ms")
print(f"Speed-up:            {t_loop.best / t_numpy.best:10.1f} x")

Python loop:            492.328 ms
NumPy vectorization:      4.861 ms
Speed-up:                 101.3 x


### Task 1: Results table
| Method | Time |
|---|---|
| Python loop | _492.328 ms_ |
| NumPy vectorization | _4.861 ms_ |

**Questions**
1. **Which was faster?** NumPy vectorization.
2. **How many times faster?** _speed-up printed above_ (typically tens to hundreds of times).
3. **Why?** NumPy runs compiled C loops over contiguous, typed memory. The Python loop has per-element interpreter overhead and creates a Python object for every element.

## Part C: Different array sizes

In [8]:
x1 = np.arange(100)
x2 = np.arange(10_000)
x3 = np.arange(1_000_000)

In [9]:
%timeit x1 ** 2

2 μs ± 169 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [10]:
%timeit x2 ** 2

15.1 μs ± 1.4 μs per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [11]:
%timeit x3 ** 2

6.12 ms ± 286 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Task 2
For small arrays (100 elements) the time is tiny (around a microsecond) and dominated by fixed overhead. As the array grows, the time increases roughly in proportion to the number of elements: 100× more elements takes roughly 100× longer (small deviations are due to memory and cache effects). _Add your own measured times here._

## Part D: Statistical calculations

In [12]:
data = np.random.normal(size=1_000_000)

In [13]:
%timeit np.mean(data)

2.04 ms ± 285 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
%timeit np.std(data)

10.2 ms ± 670 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
%timeit np.sum(data)

2.05 ms ± 98 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


**Which is fastest on my machine?** Usually `np.sum` (a single pass); `np.mean` is about the same (a sum plus one division); `np.std` is slowest because it computes the mean, the deviations and their squares, and then sums, using several passes and temporary arrays. _Confirm with your numbers._

**Why measure rather than assume?** Performance depends on hardware, library versions, data size and memory behaviour, and intuition is often wrong. Only measurement shows the truth.

## Part E: `line_profiler`
Install in the `icd341` environment (run once in a terminal, or here):

In [ ]:
%pip install line_profiler

In [ ]:
%load_ext line_profiler

from profiling_example import calculate_statistics

data = np.random.normal(size=1_000_000)
%lprun -f calculate_statistics calculate_statistics(data)

Alternatively, from a terminal in this folder:
```
kernprof -l -v profiling_example.py
```

### Task 3: Findings
Look at the **% Time** and **Time** columns of the output.
- **Line taking the most time:** normally `np.var(x)`, because it makes several passes over the data and creates temporary arrays. _Confirm with your output._
- **Time per line:** `np.sum` and `np.mean` are each a single pass and take similar time; `np.sqrt` acts on a single number and is negligible.
- **What to optimise:** the function traverses the data several times. The mean is computed twice (once directly, once inside `np.var`), so work could be shared or reduced.

## Part F: Deliberately inefficient function

In [16]:
def slow_function(x):
    result = []
    for value in x:
        result.append(value ** 2)
    total = sum(result)
    return total

def fast_function(x):
    return np.sum(x ** 2)

x = np.arange(1_000_000)
print(slow_function(x), fast_function(x))

333332833333500000 333332833333500000


In [27]:
%lprun -f slow_function slow_function(x)

UsageError: Line magic function `%lprun` not found.


In [18]:
t_slow = %timeit -o -n 1 -r 3 slow_function(x)

517 ms ± 72.6 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [19]:
t_fast = %timeit -o fast_function(x)

6.13 ms ± 357 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
print(f"slow_function: {t_slow.best*1000:.3f} ms")
print(f"fast_function: {t_fast.best*1000:.3f} ms")
print(f"Speed-up:      {t_slow.best/t_fast.best:.1f} x")

slow_function: 450.378 ms
fast_function: 5.749 ms
Speed-up:      78.3 x


### Task 4
`slow_function` builds a Python list of one million Python objects, appends to it one element at a time, and then makes a second interpreted pass with `sum`. `fast_function` squares the whole array in a single compiled operation and sums it in another, with no per-element interpreter overhead, far less memory churn, and contiguous typed data. That is why it is much more efficient.

_Caveat:_ `x ** 2` on `int64` is fine here, but very large values could overflow.

## Part G: Mini statistical computing investigation

In [22]:
np.random.seed(42)
data = np.random.normal(loc=100, scale=20, size=5_000_000)

In [23]:
def squared_deviations_loop(x):
    mean = np.mean(x)
    total = 0
    for value in x:
        total += (value - mean) ** 2
    return total

def squared_deviations_numpy(x):
    mean = np.mean(x)
    return np.sum((x - mean) ** 2)

In [24]:
t_dev_numpy = %timeit -o squared_deviations_numpy(data)

79.2 ms ± 4.36 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [25]:
# This is slow (several seconds per run), so run it only once per repeat
t_dev_loop = %timeit -o -n 1 -r 3 squared_deviations_loop(data)

4.51 s ± 184 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [26]:
print(f"Loop time : {t_dev_loop.best:.4f} s")
print(f"NumPy time: {t_dev_numpy.best:.4f} s")
print(f"Speed-up  : {t_dev_loop.best / t_dev_numpy.best:.1f} x")
print(np.isclose(squared_deviations_loop(data), squared_deviations_numpy(data)))

Loop time : 4.3297 s
NumPy time: 0.0711 s
Speed-up  : 60.9 x
True


### Questions
1. **Which is faster?** The NumPy implementation.
2. **Measured times:** _copy the loop and NumPy times printed above._
3. **Speed-up** = Loop time / NumPy time = _value printed above_ (typically hundreds of times).
4. **Why can vectorization make such a difference?** The loop executes 5 million interpreter iterations, each doing Python-level arithmetic on NumPy scalar objects. NumPy performs the subtraction, squaring and summing in optimised compiled C loops over contiguous memory (often with SIMD), with no interpreter overhead.
5. **How does profiling help?** It shows where a program actually spends its time, so you optimise the one or two lines that are the real bottleneck instead of guessing, and avoid wasting effort on code that does not matter.